## MCUNet

Outputs

- `models/mcunet_quantized.tflite` quantized (INT8 input/output, per-tensor; MicroFlow-compatible)
- `compiled_models/mcunet_quantized.mlir`

Keep `epochs` small if you only need artifacts quickly.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import tensorflow as tf

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

try:
    tf.get_logger().setLevel("ERROR")
except Exception:
    pass

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
COMPILED_DIR = REPO_ROOT / "compiled_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
COMPILED_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE = MODELS_DIR / "mcunet_quantized.tflite"
OUT_MLIR = COMPILED_DIR / "mcunet_quantized.mlir"

print("OUT_TFLITE:", OUT_TFLITE)
print("OUT_MLIR:", OUT_MLIR)

2026-03-27 12:22:47.489261: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-27 12:22:47.573231: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774610567.607609   49720 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774610567.617580   49720 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-27 12:22:47.706032: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

OUT_TFLITE: /home/nathan/Documents/ariel-os-tiny-ml/models/mcunet_quantized.tflite
OUT_MLIR: /home/nathan/Documents/ariel-os-tiny-ml/compiled_models/mcunet_quantized.mlir


In [2]:
def conv_block(x, filters, kernel_size, strides=1):
    # MicroFlow supports fused activations in CONV_2D (NONE/RELU/RELU6).
    return tf.keras.layers.Conv2D(
        filters,
        kernel_size,
        strides=strides,
        padding="same",
        activation="relu",
        use_bias=True,
    )(x)

def depthwise_separable_block(x, pointwise_filters, strides=1):
    x = tf.keras.layers.DepthwiseConv2D(
        3,
        strides=strides,
        padding="same",
        activation="relu",
        use_bias=True,
    )(x)
    x = tf.keras.layers.Conv2D(
        pointwise_filters,
        1,
        padding="same",
        activation="relu",
        use_bias=True,
    )(x)
    return x

def build_mcunet() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(32, 32, 3), batch_size=1, name="input")
    x = conv_block(inputs, 16, 3, strides=1)
    x = depthwise_separable_block(x, 32, strides=2)
    x = depthwise_separable_block(x, 64, strides=2)
    x = depthwise_separable_block(x, 64, strides=1)

    # Replace GlobalAveragePooling (-> MEAN) with AveragePooling2D (-> AVERAGE_POOL_2D).
    logits_map = tf.keras.layers.Conv2D(10, 1, padding="same", activation=None, use_bias=True)(x)
    pooled = tf.keras.layers.AveragePooling2D(pool_size=(8, 8), strides=(8, 8), padding="valid")(logits_map)
    logits = tf.keras.layers.Reshape((10,), name="logits")(pooled)
    outputs = tf.keras.layers.Softmax(name="softmax")(logits)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="mcunet")

model = build_mcunet()
model.summary()

I0000 00:00:1774610569.348355   49720 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6153 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "mcunet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (1, 32, 32, 3)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (1, 32, 32, 16)        │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (1, 16, 16, 16)        │           160 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (1, 16, 16, 32)        │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (1, 8, 8, 32)          │           320 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (1, 8, 8, 64)          │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (1, 8, 8, 64)          │           640 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (1, 8, 8, 64)          │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (1, 8, 8, 10)          │           650 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (1, 1, 1, 10)          │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Reshape)                │ (1, 10)                │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (1, 10)                │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,034 (35.29 KB)

 Trainable params: 9,034 (35.29 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
epochs = 2
batch_size = 128

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.squeeze().astype("int64")
y_test = y_test.squeeze().astype("int64")
x_train = (x_train.astype(np.float32) / 255.0)
x_test = (x_test.astype(np.float32) / 255.0)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=2,
)

print("Test accuracy:", model.evaluate(x_test, y_test, verbose=0)[1])

Epoch 1/2


I0000 00:00:1774610574.140931   49803 service.cc:148] XLA service 0x7ef39c007be0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774610574.141151   49803 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-03-27 12:22:54.187172: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1774610574.369936   49803 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1774610576.498463   49803 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


352/352 - 8s - 23ms/step - accuracy: 0.1378 - loss: 2.2122 - val_accuracy: 0.2242 - val_loss: 2.0310
Epoch 2/2
352/352 - 1s - 4ms/step - accuracy: 0.2508 - loss: 1.9523 - val_accuracy: 0.3148 - val_loss: 1.8189
Test accuracy: 0.3237000107765198


In [4]:
def representative_data_gen():
    n = min(200, x_train.shape[0])
    for i in range(n):
        yield [x_train[i : i + 1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
converter.experimental_enable_resource_variables = True

for attr in ("experimental_new_quantizer", "_experimental_new_quantizer"):
    if hasattr(converter, attr):
        setattr(converter, attr, False)

tflite_bytes = converter.convert()
OUT_TFLITE.write_bytes(tflite_bytes)
print("Wrote:", OUT_TFLITE, "bytes=", OUT_TFLITE.stat().st_size)

interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE), experimental_delegates=[])
interp.allocate_tensors()
print("Input:", interp.get_input_details()[0]["dtype"], interp.get_input_details()[0]["shape"])
print("Output:", interp.get_output_details()[0]["dtype"], interp.get_output_details()[0]["shape"])

per_channel = []
for td in interp.get_tensor_details():
    qp = td.get("quantization_parameters") or {}
    scales = qp.get("scales")
    if isinstance(scales, np.ndarray) and scales.size > 1:
        per_channel.append((td.get("name", "?"), int(scales.size)))

if per_channel:
    print("Warning: per-channel quantization detected (first 20):")
    for name, n in per_channel[:20]:
        print(" -", name, "scales=", n)
else:
    print("Per-channel quantization check: OK")

from iree.compiler import tf as tfc
from pathlib import Path as _Path
import shutil

ARTIFACTS_DIR = REPO_ROOT / "build" / "mlir_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
SAVED_MODEL_DIR = ARTIFACTS_DIR / "mcunet_quantized_saved_model"

shutil.rmtree(SAVED_MODEL_DIR, ignore_errors=True)
model.export(str(SAVED_MODEL_DIR))

mlir_bytes = tfc.compile_saved_model(
    str(SAVED_MODEL_DIR),
    import_only=True,
    exported_names=["serve"],
)
OUT_MLIR.write_bytes(mlir_bytes)

print("done writing mlir")

Saved artifact at '/tmp/tmpui4o2goj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 32, 32, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 10), dtype=tf.float32, name=None)
Captures:
  139591879827600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139589461379216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139589461378256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139589461378064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139589461378448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139589461377872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139589461379600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139589461379024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139589461379984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139589461377296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139589461380944: TensorSpec(

/home/nathan/Documents/ariel-os-tiny-ml/building/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1774610584.429777   49720 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774610584.429791   49720 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-27 12:23:04.430036: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpui4o2goj
2026-03-27 12:23:04.430557: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-27 12:23:04.430566: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpui4o2goj
I0000 00:00:1774610584.435624   49720 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
2026-03-27 12:23:04.436741: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedMo

Wrote: /home/nathan/Documents/ariel-os-tiny-ml/models/mcunet_quantized.tflite bytes= 25424
Input: <class 'numpy.int8'> [ 1 32 32  3]
Output: <class 'numpy.int8'> [ 1 10]
 - arith.constant scales= 64
 - arith.constant1 scales= 64
 - arith.constant2 scales= 64
 - arith.constant3 scales= 32
 - arith.constant4 scales= 32
 - arith.constant5 scales= 16
 - arith.constant6 scales= 10
 - arith.constant7 scales= 10
 - arith.constant8 scales= 64
 - arith.constant9 scales= 64
 - arith.constant10 scales= 64
 - arith.constant11 scales= 32
 - arith.constant12 scales= 32
 - arith.constant13 scales= 16
 - mcunet_1/conv2d_1/convolution scales= 16
 - mcunet_1/conv2d_1/Relu;mcunet_1/conv2d_1/BiasAdd;mcunet_1/conv2d_1/convolution; scales= 16
Saved artifact at '/home/nathan/Documents/ariel-os-tiny-ml/build/mlir_artifacts/mcunet_quantized_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='input')
Out

2026-03-27 12:23:05.000561: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /home/nathan/Documents/ariel-os-tiny-ml/build/mlir_artifacts/mcunet_quantized_saved_model
2026-03-27 12:23:05.001365: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /home/nathan/Documents/ariel-os-tiny-ml/build/mlir_artifacts/mcunet_quantized_saved_model
2026-03-27 12:23:05.031176: W tensorflow/compiler/mlir/tensorflow/transforms/xla_validate_inputs.cc:96] missing entry functions
